<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:OrderLine</h2>

----

### Вариант задания 15


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс OrderLine в C#, который будет представлять информацию о 
строке заказа, содержащей детали одного товара в заказе. На основе этого класса 
разработать 2-3 производных класса, демонстрирующих принципы наследования и 
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и 
методы,  а  также  переопределены  некоторые  методы  базового  класса  для 
демонстрации полиморфизма.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

interface IDiscountable
{
    decimal Discount { get; set; }
    decimal GetDiscountAmount();
}


interface IShippable
{
    string Address { get; set; }
    decimal DeliveryCost { get; set; }
    decimal GetTotalWithDelivery();
}


interface IAuditable
{
    int ChangesCount { get; }
    void LogChange(string message);
    string GetAuditInfo();
}

interface IPrintable
{
    string GetSummary();
    void Print();
}

interface IExportable
{
    string GetSummary();
    string ToCsv();
}


interface ILogger
{
    void Log(string message);
}

class ConsoleLogger : ILogger
{
    public void Log(string message)
    {
        Console.WriteLine("[LOG] " + message);
    }
}

class MemoryLogger : ILogger
{
    public List<string> Messages { get; } = new List<string>();

    public void Log(string message)
    {
        Messages.Add(message);
    }
}

interface ITaxService
{
    string Name { get; }
    decimal GetTax(decimal amount);
}

class VatTaxService : ITaxService
{
    private readonly decimal _percent;

    public VatTaxService(decimal percent)
    {
        _percent = percent;
    }

    public string Name
    {
        get { return "НДС " + _percent + "%"; }
    }

    public decimal GetTax(decimal amount)
    {
        return amount * _percent / 100;
    }
}

class ZeroTaxService : ITaxService
{
    public string Name
    {
        get { return "Без налога"; }
    }

    public decimal GetTax(decimal amount)
    {
        return 0;
    }
}

class SimpleContainer
{
    private readonly Dictionary<Type, Func<object>> _map = new Dictionary<Type, Func<object>>();

    public void RegisterSingleton<TService>(TService instance)
    {
        _map[typeof(TService)] = () => instance;
    }

    public void RegisterTransient<TService, TImpl>() where TImpl : TService, new()
    {
        _map[typeof(TService)] = () => new TImpl();
    }

    public void RegisterFactory<TService>(Func<SimpleContainer, TService> factory)
    {
        _map[typeof(TService)] = () => factory(this);
    }

    public T Resolve<T>()
    {
        if (!_map.TryGetValue(typeof(T), out Func<object> create))
        {
            throw new InvalidOperationException("Не зарегистрирован тип " + typeof(T).Name);
        }
        return (T)create();
    }
}

class OrderProcessor
{
    private readonly ILogger _logger;
    private readonly ITaxService _tax;

    public OrderProcessor(ILogger logger, ITaxService tax)
    {
        _logger = logger;
        _tax = tax;
    }

    public decimal Process(Order order)
    {
        decimal total = order.CalculateTotal();
        decimal tax = _tax.GetTax(total);
        _logger.Log($"Заказ №{order.OrderId}: сумма {total}, {_tax.Name}: {tax}");
        return total + tax;
    }
}


class OrderLine : IAuditable, IPrintable
{
    public int ProductId { get; set; }
    public string ProductName { get; set; }
    public decimal Price { get; set; }
    public string Category { get; set; } = "Общая";
    public DateTime CreatedAt { get; set; } = DateTime.Now;
    public bool IsActive { get; set; } = true;
    public decimal Weight { get; set; } = 1;
    public string Description { get; set; } = "Без описания";
    public int Rating { get; set; } = 0;
    public string Brand { get; set; } = "Без бренда";
    public List<string> Tags { get; set; } = new List<string>();

    public string Sku { get; set; } = "N/A";
    public string Currency { get; set; } = "руб.";
    public DateTime? LastModified { get; set; } = null;
    public List<string> ChangeLog { get; set; } = new List<string>();
    public ILogger Logger { get; set; }

    public OrderLine(int productId, string productName, decimal price)
    {
        ProductId = productId;
        ProductName = productName;
        Price = price;
    }

    public virtual decimal CalculateTotal()
    {
        return Price;
    }

    public virtual void UpdatePrice(decimal newPrice)
    {
        Price = newPrice;
    }

    public virtual string GetProductDetails()
    {
        return $"ID товара: {ProductId}, Название: {ProductName}, Цена: {Price} руб.";
    }

    public void Activate()
    {
        IsActive = true;
    }

    public void Deactivate()
    {
        IsActive = false;
    }

    public string GetStatus()
    {
        return $"Товар {ProductName}: категория {Category}, вес {Weight} кг, " +
               (IsActive ? "активен" : "неактивен");
    }

    public bool IsExpensive(decimal limit)
    {
        return Price > limit;
    }

    public virtual decimal GetShippingCost()
    {
        return Weight * 100;
    }

    public virtual string GetShortInfo()
    {
        return $"{ProductName}: {Price} руб.";
    }

    public string GetProductDetails(bool full)
    {
        string details = GetProductDetails();
        if (full)
        {
            details += $", Бренд: {Brand}, Рейтинг: {Rating}, Описание: {Description}";
        }
        return details;
    }

    public decimal CalculateDiscountedPrice(decimal percent)
    {
        return Price - Price * percent / 100;
    }

    public decimal CalculateDiscountedPrice(decimal percent, decimal maxDiscount)
    {
        decimal discount = Price * percent / 100;
        return Price - Math.Min(discount, maxDiscount);
    }

    public void AddTag(string tag)
    {
        Tags.Add(tag);
    }

    public void Touch(string reason)
    {
        LastModified = DateTime.Now;
        ((IAuditable)this).LogChange(reason);
        Logger?.Log($"{ProductName}: {reason}");
    }

    public string GetPriceWithCurrency()
    {
        return Price + " " + Currency;
    }

    public bool HasTag(string tag)
    {
        return Tags.Contains(tag);
    }

    public bool IsNew(int days)
    {
        return (DateTime.Now - CreatedAt).TotalDays <= days;
    }

    int IAuditable.ChangesCount
    {
        get { return ChangeLog.Count; }
    }

    void IAuditable.LogChange(string message)
    {
        ChangeLog.Add($"{DateTime.Now:HH:mm:ss} {message}");
    }

    string IAuditable.GetAuditInfo()
    {
        return $"{ProductName}: изменений {ChangeLog.Count}, последнее: " +
               (LastModified?.ToString("dd.MM.yyyy HH:mm") ?? "нет");
    }

    string IPrintable.GetSummary()
    {
        return GetShortInfo();
    }

    void IPrintable.Print()
    {
        Console.WriteLine("Печать: " + GetShortInfo());
    }
}


class StandardLine : OrderLine
{
    public int Units { get; set; }
    public string Warehouse { get; set; } = "Основной склад";
    public bool IsReserved { get; set; } = false;
    public int MinUnits { get; set; } = 1;
    public string Supplier { get; set; } = "Не указан";
    public int PackageCount { get; set; } = 1;
    public int DeliveryDays { get; set; } = 3;
    public bool IsFragile { get; set; } = false;
    public string Barcode { get; set; } = "0000000000";

    public string ShelfCode { get; set; } = "Не указана";
    public int ReorderLevel { get; set; } = 5;
    public DateTime? LastInventory { get; set; } = null;
    public bool IsBackordered { get; set; } = false;

    public StandardLine(int productId, string productName, decimal price, int units)
        : base(productId, productName, price)
    {
        Units = units;
    }

    public override decimal CalculateTotal()
    {
        return Price * Units;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Количество: {Units} шт.";
    }

    public void IncreaseUnits(int count)
    {
        Units += count;
    }

    public void DecreaseUnits(int count)
    {
        Units = Math.Max(MinUnits, Units - count);
    }

    public void Reserve()
    {
        IsReserved = true;
    }

    public string GetWarehouseInfo()
    {
        return $"Склад: {Warehouse}, поставщик: {Supplier}, " +
               (IsReserved ? "в резерве" : "без резерва");
    }

    public override decimal GetShippingCost()
    {
        return Weight * Units * 100 + (IsFragile ? 200 : 0);
    }

    public override string GetShortInfo()
    {
        return base.GetShortInfo() + $" x {Units} шт.";
    }

    public decimal CalculateTotal(int units)
    {
        return Price * units;
    }

    public string GetDeliveryInfo()
    {
        return $"Доставка: {DeliveryDays} дн., упаковок: {PackageCount}, " +
               (IsFragile ? "хрупкий груз" : "обычный груз");
    }

    public bool NeedsReorder()
    {
        return Units <= ReorderLevel;
    }

    public void Backorder()
    {
        IsBackordered = true;
        Touch("Оформлен предзаказ");
    }

    public void MakeInventory()
    {
        LastInventory = DateTime.Now;
        Touch("Проведена инвентаризация");
    }

    public string GetStockInfo()
    {
        return $"Полка {ShelfCode}, остаток {Units} шт., " +
               (NeedsReorder() ? "нужен дозаказ" : "запас достаточен");
    }
}


class SpecialLine : OrderLine
{
    public decimal Discount { get; set; }
    public string PromoCode { get; set; } = "NONE";
    public DateTime PromoEnd { get; set; } = DateTime.Now.AddDays(30);
    public bool IsPromoActive { get; set; } = true;
    public decimal MaxDiscount { get; set; } = 50;
    public string Season { get; set; } = "Осень";
    public bool IsLimited { get; set; } = false;
    public int StockLeft { get; set; } = 100;
    public string Badge { get; set; } = "SALE";

    public string Campaign { get; set; } = "Без кампании";
    public int MinPurchase { get; set; } = 1;
    public int PromoLimit { get; set; } = 100;
    public bool IsExclusive { get; set; } = false;

    public SpecialLine(int productId, string productName, decimal price, decimal discount)
        : base(productId, productName, price)
    {
        Discount = discount;
    }

    public override void UpdatePrice(decimal newPrice)
    {
        Price = newPrice - newPrice * Discount / 100;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Скидка: {Discount}%";
    }

    public void ExtendPromo(int days)
    {
        PromoEnd = PromoEnd.AddDays(days);
    }

    public void ClosePromo()
    {
        IsPromoActive = false;
    }

    public decimal GetDiscountAmount(decimal basePrice)
    {
        return basePrice * Math.Min(Discount, MaxDiscount) / 100;
    }

    public string GetPromoInfo()
    {
        return $"Промокод: {PromoCode}, действует до {PromoEnd:dd.MM.yyyy}, " +
               (IsPromoActive ? "активен" : "закрыт");
    }

    public override decimal GetShippingCost()
    {
        return base.GetShippingCost() / 2;
    }

    public override string GetShortInfo()
    {
        return base.GetShortInfo() + $" [{Badge}, -{Discount}%]";
    }

    public void UpdatePrice(decimal newPrice, decimal extraDiscount)
    {
        Price = newPrice - newPrice * (Discount + extraDiscount) / 100;
    }

    public bool Sell()
    {
        return Sell(1);
    }

    public bool Sell(int count)
    {
        if (count > StockLeft)
        {
            return false;
        }
        StockLeft -= count;
        return true;
    }

    public bool IsPromoValid()
    {
        return IsPromoActive && DateTime.Now <= PromoEnd && StockLeft > 0;
    }

    public decimal GetFinalPrice()
    {
        return Price - Price * Math.Min(Discount, MaxDiscount) / 100;
    }

    public void MakeExclusive()
    {
        IsExclusive = true;
        Badge = "VIP";
        Touch("Товар стал эксклюзивным");
    }

    public string GetCampaignInfo()
    {
        return $"Кампания: {Campaign}, мин. покупка: {MinPurchase} шт., " +
               (IsExclusive ? "эксклюзив" : "обычный товар");
    }
}


class FreeLine : OrderLine
{
    public decimal Prepayment { get; set; }
    public bool IsPrepaid { get; set; } = false;
    public DateTime? PaymentDate { get; set; } = null;
    public string PaymentMethod { get; set; } = "Не выбран";
    public DateTime DueDate { get; set; } = DateTime.Now.AddDays(7);
    public string Manager { get; set; } = "Не назначен";
    public string Comment { get; set; } = "";
    public int PaymentsCount { get; set; } = 0;

    public int Installments { get; set; } = 1;
    public string Guarantor { get; set; } = "Нет";
    public decimal PenaltyPercent { get; set; } = 1;
    public bool IsClosed { get; set; } = false;

    public FreeLine(int productId, string productName, decimal price, decimal prepayment)
        : base(productId, productName, price)
    {
        Prepayment = prepayment;
    }

    public override decimal CalculateTotal()
    {
        return Price - Prepayment;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Предварительный платеж: {Prepayment} руб.";
    }

    public void PayPrepayment(string method)
    {
        PaymentMethod = method;
        PaymentDate = DateTime.Now;
        IsPrepaid = true;
    }

    public decimal GetRemainder()
    {
        return Price - Prepayment;
    }

    public bool IsFullyPaid()
    {
        return Prepayment >= Price;
    }

    public string GetPaymentInfo()
    {
        return IsPrepaid
            ? $"Предоплата внесена {PaymentDate:dd.MM.yyyy}, способ: {PaymentMethod}"
            : "Предоплата не внесена";
    }

    public override decimal GetShippingCost()
    {
        return 0;
    }

    public override string GetShortInfo()
    {
        return base.GetShortInfo() + $", предоплата {Prepayment} руб., срок до {DueDate:dd.MM.yyyy}";
    }

    public void PayPrepayment(string method, decimal amount)
    {
        Prepayment += amount;
        PaymentsCount++;
        PayPrepayment(method);
    }

    public void ExtendDueDate(int days)
    {
        DueDate = DueDate.AddDays(days);
    }

    public decimal GetInstallmentAmount()
    {
        return Installments > 0 ? GetRemainder() / Installments : 0;
    }

    public bool IsOverdue()
    {
        return !IsFullyPaid() && DateTime.Now > DueDate;
    }

    public int GetDaysLeft()
    {
        return (int)(DueDate - DateTime.Now).TotalDays;
    }

    public void SetGuarantor(string name)
    {
        Guarantor = name;
        Touch("Назначен поручитель: " + name);
    }
}


class PremiumLine : StandardLine
{
    public int WarrantyMonths { get; set; }
    public string SupportLevel { get; set; } = "Стандартная";
    public bool IsGiftWrapped { get; set; } = false;

    public string ServiceCenter { get; set; } = "Не указан";
    public decimal InsuranceCost { get; set; } = 0;
    public string Tier { get; set; } = "Basic";
    public bool HasExtendedSupport { get; set; } = false;

    public PremiumLine(int productId, string productName, decimal price, int units, int warrantyMonths)
        : base(productId, productName, price, units)
    {
        WarrantyMonths = warrantyMonths;
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Гарантия: {WarrantyMonths} мес.";
    }

    public void ExtendWarranty(int months)
    {
        WarrantyMonths += months;
    }

    public void WrapAsGift()
    {
        IsGiftWrapped = true;
    }

    public string GetWarrantyInfo()
    {
        return $"Гарантия {WarrantyMonths} мес., поддержка: {SupportLevel}, " +
               (IsGiftWrapped ? "подарочная упаковка" : "обычная упаковка");
    }

    public override decimal GetShippingCost()
    {
        return base.GetShippingCost() / 2;
    }

    public override string GetShortInfo()
    {
        return base.GetShortInfo() + $", гарантия {WarrantyMonths} мес.";
    }

    public DateTime GetWarrantyEnd(DateTime start)
    {
        return start.AddMonths(WarrantyMonths);
    }

    public void AddInsurance(decimal cost)
    {
        InsuranceCost += cost;
        HasExtendedSupport = true;
        Touch("Добавлена страховка: " + cost + " руб.");
    }

    public void UpgradeTier(string tier)
    {
        Tier = tier;
        Touch("Уровень изменён на " + tier);
    }

    public decimal GetTotalWithInsurance()
    {
        return CalculateTotal() + InsuranceCost;
    }
}


class CombinedLine : OrderLine, IDiscountable, IShippable, IExportable, IPrintable
{
    public int Units { get; set; }
    public decimal Discount { get; set; }
    public string Address { get; set; }
    public decimal DeliveryCost { get; set; }

    public string Courier { get; set; } = "Не назначен";
    public string TrackingNumber { get; set; } = "Нет";
    public DateTime DeliveryDate { get; set; } = DateTime.Now.AddDays(5);
    public string GiftMessage { get; set; } = "";

    public CombinedLine(int productId, string productName, decimal price, int units,
                        decimal discount, string address, decimal deliveryCost)
        : base(productId, productName, price)
    {
        Units = units;
        Discount = discount;
        Address = address;
        DeliveryCost = deliveryCost;
    }

    public decimal GetDiscountAmount()
    {
        return Price * Units * Discount / 100;
    }

    public decimal GetTotalWithDelivery()
    {
        return CalculateTotal() + DeliveryCost;
    }

    public override decimal CalculateTotal()
    {
        return Price * Units - GetDiscountAmount();
    }

    public override string GetProductDetails()
    {
        return base.GetProductDetails() +
               $", Количество: {Units} шт., Скидка: {Discount}%, Адрес: {Address}";
    }

    public override decimal GetShippingCost()
    {
        return DeliveryCost;
    }

    public override string GetShortInfo()
    {
        return base.GetShortInfo() + $" x {Units} шт., доставка: {Address}";
    }

    public void AssignCourier(string courier, string trackingNumber)
    {
        Courier = courier;
        TrackingNumber = trackingNumber;
        Touch("Назначен курьер: " + courier);
    }

    public string GetTrackingInfo()
    {
        return $"Курьер: {Courier}, трек-номер: {TrackingNumber}, доставка до {DeliveryDate:dd.MM.yyyy}";
    }

    public bool IsDeliveryLate()
    {
        return DateTime.Now > DeliveryDate;
    }

    public void SetGiftMessage(string message)
    {
        GiftMessage = message;
    }

    string IPrintable.GetSummary()
    {
        return $"{ProductName} x {Units} шт. для печати, адрес: {Address}";
    }

    void IPrintable.Print()
    {
        Console.WriteLine("Печать накладной: " + GetShortInfo());
    }

    string IExportable.GetSummary()
    {
        return $"{ProductId};{ProductName}";
    }

    string IExportable.ToCsv()
    {
        return $"{ProductId};{ProductName};{Units};{CalculateTotal()}";
    }
}


class Order
{
    private int _orderId;
    private Customer _customer;
    private List<OrderLine> _lines;

    public Order(int orderId, Customer customer)
    {
        _orderId = orderId;
        _customer = customer;
        _lines = new List<OrderLine>();
    }

    public int OrderId
    {
        get { return _orderId; }
        set { _orderId = value; }
    }

    public Customer Customer
    {
        get { return _customer; }
        set { _customer = value; }
    }

    public List<OrderLine> Lines
    {
        get { return _lines; }
        set { _lines = value; }
    }

    public void AddLine(OrderLine line)
    {
        _lines.Add(line);
    }

    public decimal CalculateTotal()
    {
        decimal total = 0;
        foreach (OrderLine line in _lines)
        {
            total += line.CalculateTotal();
        }
        return total;
    }

    public string GetOrderDetails()
    {
        string result = $"Заказ №{OrderId}, клиент: {Customer.Name}";
        foreach (OrderLine line in _lines)
        {
            result += Environment.NewLine + "  " + line.GetProductDetails();
        }
        return result;
    }

    public decimal GetShippingTotal()
    {
        decimal total = 0;
        foreach (OrderLine line in _lines)
        {
            total += line.GetShippingCost();
        }
        return total;
    }
}


class Customer
{
    private int _customerId;
    private string _name;
    private List<Order> _orders;

    public Customer(int customerId, string name)
    {
        _customerId = customerId;
        _name = name;
        _orders = new List<Order>();
    }

    public int CustomerId
    {
        get { return _customerId; }
        set { _customerId = value; }
    }

    public string Name
    {
        get { return _name; }
        set { _name = value; }
    }

    public List<Order> Orders
    {
        get { return _orders; }
        set { _orders = value; }
    }

    public void AddOrder(Order order)
    {
        order.Customer = this;
        _orders.Add(order);
    }

    public decimal GetTotalSpent()
    {
        decimal total = 0;
        foreach (Order order in _orders)
        {
            total += order.CalculateTotal();
        }
        return total;
    }
}


class LineRepository<T> where T : OrderLine
{
    private List<T> _items = new List<T>();

    public int Count
    {
        get { return _items.Count; }
    }

    public List<T> Items
    {
        get { return _items; }
    }

    public void Add(T item)
    {
        _items.Add(item);
    }

    public bool Remove(int productId)
    {
        T item = FindById(productId);
        if (item == null)
        {
            return false;
        }
        return _items.Remove(item);
    }

    public T FindById(int productId)
    {
        foreach (T item in _items)
        {
            if (item.ProductId == productId)
            {
                return item;
            }
        }
        return null;
    }

    public List<T> Filter(Func<T, bool> condition)
    {
        List<T> result = new List<T>();
        foreach (T item in _items)
        {
            if (condition(item))
            {
                result.Add(item);
            }
        }
        return result;
    }

    public decimal GetTotal()
    {
        decimal total = 0;
        foreach (T item in _items)
        {
            total += item.CalculateTotal();
        }
        return total;
    }

    public T GetMostExpensive()
    {
        T best = null;
        foreach (T item in _items)
        {
            if (best == null || item.Price > best.Price)
            {
                best = item;
            }
        }
        return best;
    }
}


class Pair<TFirst, TSecond>
{
    public TFirst First { get; set; }
    public TSecond Second { get; set; }

    public Pair(TFirst first, TSecond second)
    {
        First = first;
        Second = second;
    }

    public Pair<TSecond, TFirst> Swap()
    {
        return new Pair<TSecond, TFirst>(Second, First);
    }

    public override string ToString()
    {
        return $"({First}, {Second})";
    }
}


OrderLine standard = new StandardLine(
    1, "Ноутбук", 50000, 2);

OrderLine special = new SpecialLine(
    2, "Телефон", 30000, 10);

OrderLine free = new FreeLine(
    3, "Наушники", 5000, 1000);


Console.WriteLine("СТАНДАРТНАЯ СТРОКА");
Console.WriteLine(standard.GetProductDetails());
Console.WriteLine("Общая стоимость: " +
                  standard.CalculateTotal() + " руб.");

Console.WriteLine();


Console.WriteLine("СПЕЦИАЛЬНАЯ СТРОКА");
Console.WriteLine(special.GetProductDetails());

special.UpdatePrice(30000);

Console.WriteLine("Цена со скидкой: " +
                  special.Price + " руб.");

Console.WriteLine();


Console.WriteLine("БЕСПЛАТНАЯ СТРОКА");
Console.WriteLine(free.GetProductDetails());
Console.WriteLine("Общая стоимость после вычета предоплаты: " +
                  free.CalculateTotal() + " руб.");

Console.WriteLine();

Customer customer = new Customer(1, "Иван Петров");
Order order = new Order(100, customer);
order.AddLine(standard);
order.AddLine(special);
order.AddLine(free);
customer.AddOrder(order);

Console.WriteLine("ЗАКАЗ");
Console.WriteLine(order.GetOrderDetails());
Console.WriteLine("Стоимость заказа: " +
                  order.CalculateTotal() + " руб.");
Console.WriteLine("Всего потрачено клиентом " + customer.Name + ": " +
                  customer.GetTotalSpent() + " руб.");

Console.WriteLine();

Console.WriteLine("ПРОСТОЕ НАСЛЕДОВАНИЕ");
StandardLine standardLine = (StandardLine)standard;
standardLine.Warehouse = "Склад №2";
standardLine.Supplier = "ТехноОпт";
standardLine.Reserve();
standardLine.IncreaseUnits(3);
Console.WriteLine(standardLine.GetStatus());
Console.WriteLine(standardLine.GetWarehouseInfo());
Console.WriteLine("Количество: " + standardLine.Units +
                  ", стоимость: " + standardLine.CalculateTotal() + " руб.");

SpecialLine specialLine = (SpecialLine)special;
specialLine.PromoCode = "AUTUMN10";
specialLine.ExtendPromo(15);
Console.WriteLine(specialLine.GetPromoInfo());
Console.WriteLine("Размер скидки от 30000: " +
                  specialLine.GetDiscountAmount(30000) + " руб.");

FreeLine freeLine = (FreeLine)free;
Console.WriteLine(freeLine.GetPaymentInfo());
freeLine.PayPrepayment("Карта");
Console.WriteLine(freeLine.GetPaymentInfo());
Console.WriteLine("Остаток к оплате: " + freeLine.GetRemainder() + " руб.");

Console.WriteLine();

Console.WriteLine("МНОГОУРОВНЕВОЕ НАСЛЕДОВАНИЕ");
PremiumLine premium = new PremiumLine(
    4, "Планшет", 40000, 1, 12);
premium.ExtendWarranty(6);
premium.WrapAsGift();
premium.SupportLevel = "VIP";
premium.Category = "Электроника";
Console.WriteLine(premium.GetProductDetails());
Console.WriteLine(premium.GetWarrantyInfo());
Console.WriteLine(premium.GetStatus());
Console.WriteLine("Общая стоимость: " + premium.CalculateTotal() + " руб.");

Console.WriteLine();

Console.WriteLine("МНОЖЕСТВЕННОЕ НАСЛЕДОВАНИЕ");
CombinedLine combined = new CombinedLine(
    5, "Монитор", 15000, 2, 10, "г. Екатеринбург, ул. Ленина, 1", 500);
Console.WriteLine(combined.GetProductDetails());
Console.WriteLine("Скидка: " + combined.GetDiscountAmount() + " руб.");
Console.WriteLine("Стоимость: " + combined.CalculateTotal() + " руб.");
Console.WriteLine("Стоимость с доставкой: " +
                  combined.GetTotalWithDelivery() + " руб.");

IDiscountable discountable = combined;
IShippable shippable = combined;
Console.WriteLine("Через IDiscountable: " + discountable.GetDiscountAmount() + " руб.");
Console.WriteLine("Через IShippable: " + shippable.Address);

order.AddLine(premium);
order.AddLine(combined);
Console.WriteLine();
Console.WriteLine("Стоимость заказа после добавления новых строк: " +
                  order.CalculateTotal() + " руб.");

Console.WriteLine();

Console.WriteLine("ПЕРЕГРУЗКА МЕТОДОВ");
Console.WriteLine("Цена со скидкой 10%: " +
                  standard.CalculateDiscountedPrice(10) + " руб.");
Console.WriteLine("Цена со скидкой 10% (не больше 3000 руб.): " +
                  standard.CalculateDiscountedPrice(10, 3000) + " руб.");
Console.WriteLine(standard.GetProductDetails());
standard.Brand = "TechPro";
standard.Rating = 5;
standard.Description = "Мощный ноутбук";
Console.WriteLine(standard.GetProductDetails(true));
Console.WriteLine("Стоимость 2 шт.: " + standardLine.CalculateTotal(2) + " руб.");
Console.WriteLine("Стоимость всех " + standardLine.Units + " шт.: " +
                  standardLine.CalculateTotal() + " руб.");
specialLine.UpdatePrice(30000, 5);
Console.WriteLine("Цена с дополнительной скидкой 5%: " + specialLine.Price + " руб.");
specialLine.Sell();
specialLine.Sell(10);
Console.WriteLine("Осталось на складе: " + specialLine.StockLeft + " шт.");
freeLine.PayPrepayment("Наличные", 500);
Console.WriteLine("Предоплата: " + freeLine.Prepayment + " руб., платежей: " +
                  freeLine.PaymentsCount + ", способ: " + freeLine.PaymentMethod);

Console.WriteLine();

Console.WriteLine("ПЕРЕКРЫТИЕ МЕТОДОВ И ПОЛИМОРФИЗМ");
List<OrderLine> lines = new List<OrderLine> { standard, special, free, premium, combined };
foreach (OrderLine item in lines)
{
    Console.WriteLine(item.GetShortInfo());
    Console.WriteLine("  Доставка: " + item.GetShippingCost() +
                      " руб., стоимость: " + item.CalculateTotal() + " руб.");
}
Console.WriteLine("Стоимость доставки всего заказа: " + order.GetShippingTotal() + " руб.");

Console.WriteLine();

Console.WriteLine("GENERIC КЛАССЫ");
LineRepository<OrderLine> repository = new LineRepository<OrderLine>();
foreach (OrderLine item in lines)
{
    repository.Add(item);
}
Console.WriteLine("Строк в репозитории: " + repository.Count);
Console.WriteLine("Общая стоимость: " + repository.GetTotal() + " руб.");
Console.WriteLine("Самая дорогая строка: " + repository.GetMostExpensive().ProductName);
List<OrderLine> cheap = repository.Filter(l => l.Price < 20000);
Console.WriteLine("Строк дешевле 20000 руб.: " + cheap.Count);
Console.WriteLine("Поиск по ID 3: " + repository.FindById(3).ProductName);
Console.WriteLine("Удаление ID 3: " + repository.Remove(3) + ", осталось: " + repository.Count);

LineRepository<StandardLine> standardRepository = new LineRepository<StandardLine>();
standardRepository.Add(standardLine);
standardRepository.Add(premium);
Console.WriteLine("Стандартных строк: " + standardRepository.Count +
                  ", сумма: " + standardRepository.GetTotal() + " руб.");

Pair<string, decimal> namePrice = new Pair<string, decimal>(standard.ProductName, standard.Price);
Console.WriteLine("Пара: " + namePrice);
Console.WriteLine("После Swap: " + namePrice.Swap());


Console.WriteLine();
Console.WriteLine("НОВЫЕ АТРИБУТЫ И МЕТОДЫ");

standardLine.ShelfCode = "A-12";
standardLine.ReorderLevel = 6;
Console.WriteLine(standardLine.GetStockInfo());
standardLine.MakeInventory();

specialLine.Campaign = "Чёрная пятница";
specialLine.MakeExclusive();
Console.WriteLine(specialLine.GetCampaignInfo());
Console.WriteLine("Акция действует: " + specialLine.IsPromoValid());
Console.WriteLine("Итоговая цена с учётом лимита скидки: " + specialLine.GetFinalPrice() + " руб.");

freeLine.Installments = 3;
freeLine.SetGuarantor("Пётр Сидоров");
Console.WriteLine("Платёж по рассрочке: " + freeLine.GetInstallmentAmount() + " руб.");
Console.WriteLine("Просрочено: " + freeLine.IsOverdue() + ", дней до срока: " + freeLine.GetDaysLeft());

premium.ServiceCenter = "СЦ Екатеринбург";
premium.UpgradeTier("Gold");
premium.AddInsurance(2000);
Console.WriteLine("Гарантия до: " + premium.GetWarrantyEnd(DateTime.Today).ToString("dd.MM.yyyy"));
Console.WriteLine("Итого со страховкой: " + premium.GetTotalWithInsurance() + " руб.");

combined.AssignCourier("СДЭК", "TRK-001");
combined.SetGiftMessage("С днём рождения!");
Console.WriteLine(combined.GetTrackingInfo());
Console.WriteLine("Доставка просрочена: " + combined.IsDeliveryLate());

Console.WriteLine();
Console.WriteLine("ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ");

IAuditable auditable = standard;
auditable.LogChange("Проверка аудита");
Console.WriteLine("Изменений: " + auditable.ChangesCount);
Console.WriteLine(auditable.GetAuditInfo());

IAuditable premiumAudit = premium;
Console.WriteLine(premiumAudit.GetAuditInfo());

foreach (OrderLine item in lines)
{
    ((IPrintable)item).Print();
}

Console.WriteLine("IPrintable.GetSummary: " + ((IPrintable)combined).GetSummary());
Console.WriteLine("IExportable.GetSummary: " + ((IExportable)combined).GetSummary());
Console.WriteLine("CSV: " + ((IExportable)combined).ToCsv());

Console.WriteLine();
Console.WriteLine("УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ");

var container = new SimpleContainer();
container.RegisterSingleton<ILogger>(new ConsoleLogger());
container.RegisterSingleton<ITaxService>(new VatTaxService(20));
container.RegisterFactory<OrderProcessor>(c =>
    new OrderProcessor(c.Resolve<ILogger>(), c.Resolve<ITaxService>()));

OrderProcessor processor = container.Resolve<OrderProcessor>();
decimal totalWithTax = processor.Process(order);
Console.WriteLine("Итого с налогом: " + totalWithTax + " руб.");

foreach (OrderLine item in lines)
{
    item.Logger = container.Resolve<ILogger>();
}
standard.Touch("Обновлено описание");

var memoryLogger = new MemoryLogger();
var testProcessor = new OrderProcessor(memoryLogger, new ZeroTaxService());
decimal totalNoTax = testProcessor.Process(order);
Console.WriteLine("Итого без налога: " + totalNoTax + " руб.");
Console.WriteLine("Сообщений в MemoryLogger: " + memoryLogger.Messages.Count);

СТАНДАРТНАЯ СТРОКА
ID товара: 1, Название: Ноутбук, Цена: 50000 руб., Количество: 2 шт.
Общая стоимость: 100000 руб.

СПЕЦИАЛЬНАЯ СТРОКА
ID товара: 2, Название: Телефон, Цена: 30000 руб., Скидка: 10%
Цена со скидкой: 27000 руб.

БЕСПЛАТНАЯ СТРОКА
ID товара: 3, Название: Наушники, Цена: 5000 руб., Предварительный платеж: 1000 руб.
Общая стоимость после вычета предоплаты: 4000 руб.

ЗАКАЗ
Заказ №100, клиент: Иван Петров
  ID товара: 1, Название: Ноутбук, Цена: 50000 руб., Количество: 2 шт.
  ID товара: 2, Название: Телефон, Цена: 27000 руб., Скидка: 10%
  ID товара: 3, Название: Наушники, Цена: 5000 руб., Предварительный платеж: 1000 руб.
Стоимость заказа: 131000 руб.
Всего потрачено клиентом Иван Петров: 131000 руб.

ПРОСТОЕ НАСЛЕДОВАНИЕ
Товар Ноутбук: категория Общая, вес 1 кг, активен
Склад: Склад №2, поставщик: ТехноОпт, в резерве
Количество: 5, стоимость: 250000 руб.
Промокод: AUTUMN10, действует до 12.12.2026, активен
Размер скидки от 30000: 3000 руб.
Предоплата не внесена
Предопла